# Multi-Agent Sepsis Prediction — All 6 Experiments

This notebook runs all 6 experimental versions (v1-v6) systematically and saves results for comparison.

| Version | Patients | LR | Hidden/Layers | Dropout | Focal a | Expected |
|---------|----------|----|---------------|---------|---------|----------|
| v1 | 725 | 1e-3 | 64/2 | 0.3 | 0.25 | ~0.74 (baseline) |
| v2 | 3,559 | 1e-3 | 64/2 | 0.3 | 0.25 | ~0.67 (fails) |
| v3 | 3,559 | 1e-4 | 64/2 | 0.3 | 0.25 | ~0.71 (winner) |
| v4 | 3,559 | 1e-4 | 64/2 | 0.3 | 0.35 | ~0.69 |
| v5 | 3,559 | 1e-4 | 64/2 | 0.4 | 0.25 | ~0.72 |
| v6 | 3,559 | 1e-4 | 32/1 | 0.3 | 0.25 | ~0.72 |

**Author:** Jason | **Date:** March 2026

**Runtime:** ~2-3 hours total on T4 GPU. Each version saves independently so you can resume if disconnected.

## 1. Setup

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm h5py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report
)

# Paths
PROJECT_PATH = '/content/drive/MyDrive/Sepsis'
DATA_PATH = f'{PROJECT_PATH}/data/processed/mimic_harmonized'
MODEL_PATH = f'{PROJECT_PATH}/models'

sys.path.append(f'{PROJECT_PATH}/src')
os.makedirs(MODEL_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from models.multi_agent import MultiAgentSepsisPredictor, FocalLoss, count_parameters
print('Model imported successfully!')

## 2. Experiment Configurations

In [ ]:
# Feature definitions
VITALS_FEATURES = ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat']
LABS_FEATURES = ['bun', 'chloride', 'creatinine', 'wbc', 'bicarbonate', 'platelets',
                 'magnesium', 'calcium', 'potassium', 'sodium', 'glucose',
                 'fio2', 'ph', 'paco2', 'pao2', 'lactate', 'bilirubin']
ALL_FEATURES = VITALS_FEATURES + LABS_FEATURES

# ============================================================
# ALL 6 EXPERIMENT CONFIGS
# ============================================================
EXPERIMENTS = {
    'v1': {
        'description': 'Baseline - 725 patients, LR=1e-3',
        'num_patients': 725,
        'learning_rate': 1e-3,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.3,
        'focal_alpha': 0.25,
    },
    'v2': {
        'description': 'Full data, SAME LR (expected to fail)',
        'num_patients': None,  # use all 3,559
        'learning_rate': 1e-3,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.3,
        'focal_alpha': 0.25,
    },
    'v3': {
        'description': 'Full data, REDUCED LR (expected winner)',
        'num_patients': None,
        'learning_rate': 1e-4,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.3,
        'focal_alpha': 0.25,
    },
    'v4': {
        'description': 'Full data, adjusted focal alpha',
        'num_patients': None,
        'learning_rate': 1e-4,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.3,
        'focal_alpha': 0.35,
    },
    'v5': {
        'description': 'Full data, higher dropout',
        'num_patients': None,
        'learning_rate': 1e-4,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.4,
        'focal_alpha': 0.25,
    },
    'v6': {
        'description': 'Full data, smaller model',
        'num_patients': None,
        'learning_rate': 1e-4,
        'hidden_dim': 32,
        'num_layers': 1,
        'dropout': 0.3,
        'focal_alpha': 0.25,
    },
}

# Shared across all experiments
SHARED = {
    'sequence_length': 24,
    'batch_size': 32,
    'weight_decay': 1e-4,
    'epochs': 50,
    'patience': 10,
    'focal_gamma': 2.0,
    'random_seed': 42,
    'test_size': 0.2,
    'val_size': 0.1,
}

print(f'Defined {len(EXPERIMENTS)} experiments:')
for name, cfg in EXPERIMENTS.items():
    pts = cfg['num_patients'] or 'all (3,559)'
    print(f"  {name}: {cfg['description']}  [LR={cfg['learning_rate']}, H={cfg['hidden_dim']}, L={cfg['num_layers']}, D={cfg['dropout']}, a={cfg['focal_alpha']}]")

## 3. Dataset Class

In [ ]:
class SepsisSequenceDataset(Dataset):
    """Creates 24-hour sliding window sequences for training."""

    def __init__(self, df, vitals_features, labs_features, seq_length=24):
        self.seq_length = seq_length
        self.vitals_features = vitals_features
        self.labs_features = labs_features
        self.all_features = vitals_features + labs_features

        self.sequences = []
        self.labels = []
        self.patient_ids = []

        for patient_id, group in tqdm(df.groupby('subject_id'), desc='Building sequences'):
            group = group.sort_values('charttime').reset_index(drop=True)
            if len(group) < seq_length:
                continue
            for i in range(seq_length, len(group) + 1):
                seq = group.iloc[i - seq_length:i]
                self.sequences.append(seq[self.all_features].values)
                self.labels.append(seq['sepsis_label'].iloc[-1])
                self.patient_ids.append(patient_id)

        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)

        # Missing mask BEFORE filling NaN
        self.missing_mask = np.isnan(self.sequences)
        self.sequences = np.nan_to_num(self.sequences, nan=0.0)

        print(f'  -> {len(self.sequences):,} sequences, {self.labels.mean()*100:.1f}% positive')

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        n_vitals = len(self.vitals_features)
        return {
            'vitals': torch.tensor(self.sequences[idx, :, :n_vitals], dtype=torch.float32),
            'labs': torch.tensor(self.sequences[idx, :, n_vitals:], dtype=torch.float32),
            'labs_mask': torch.tensor(self.missing_mask[idx, :, n_vitals:], dtype=torch.float32),
            'all_features': torch.tensor(self.sequences[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32).unsqueeze(0),
        }

## 4. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='Train', leave=False):
        vitals = batch['vitals'].to(device)
        labs = batch['labs'].to(device)
        labs_mask = batch['labs_mask'].to(device)
        all_features = batch['all_features'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        output = model(vitals, labs, labs_mask, all_features)
        loss = criterion(output['logits'], labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(output['probability'].detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    return (
        total_loss / len(loader),
        roc_auc_score(all_labels, all_preds),
        average_precision_score(all_labels, all_preds),
    )


def evaluate(model, loader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_weights = [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Eval', leave=False):
            vitals = batch['vitals'].to(device)
            labs = batch['labs'].to(device)
            labs_mask = batch['labs_mask'].to(device)
            all_features = batch['all_features'].to(device)
            labels = batch['label'].to(device)

            output = model(vitals, labs, labs_mask, all_features)
            loss = criterion(output['logits'], labels)

            total_loss += loss.item()
            all_preds.extend(output['probability'].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_weights.extend(output['agent_weights'].cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    return (
        total_loss / len(loader),
        roc_auc_score(all_labels, all_preds),
        average_precision_score(all_labels, all_preds),
        all_preds,
        all_labels,
        np.array(all_weights),
    )

## 5. Load Data

In [ ]:
# Load once — we'll subset for v1
print('Loading data...')
df_raw = pd.read_hdf(f'{DATA_PATH}/mimic_processed_large.h5')

# Check available features
vitals_available = [f for f in VITALS_FEATURES if f in df_raw.columns]
labs_available = [f for f in LABS_FEATURES if f in df_raw.columns]

print(f'Loaded: {len(df_raw):,} rows, {df_raw["subject_id"].nunique()} patients')
print(f'Features: {len(vitals_available)} vitals, {len(labs_available)} labs')

if len(vitals_available) < len(VITALS_FEATURES):
    print(f'  Missing vitals: {set(VITALS_FEATURES) - set(vitals_available)}')
if len(labs_available) < len(LABS_FEATURES):
    print(f'  Missing labs: {set(LABS_FEATURES) - set(labs_available)}')

# Update to what's available
VITALS_FEATURES = vitals_available
LABS_FEATURES = labs_available
ALL_FEATURES = vitals_available + labs_available
print(f'Using {len(ALL_FEATURES)} total features')

## 6. Data Preparation Helper

In [ ]:
def prepare_data(df_raw, num_patients, seed=42):
    """
    Prepare train/val/test splits for one experiment.
    If num_patients is set, subsample first (for v1).
    Returns normalized train_df, val_df, test_df and feature_stats.
    """
    df = df_raw.copy()

    # Subsample patients if needed (v1 uses 725)
    all_patient_ids = df['subject_id'].unique()
    if num_patients is not None and num_patients < len(all_patient_ids):
        patient_labels_all = df.groupby('subject_id')['sepsis_label'].max()
        selected_ids, _ = train_test_split(
            all_patient_ids,
            train_size=num_patients,
            stratify=patient_labels_all.loc[all_patient_ids],
            random_state=seed
        )
        df = df[df['subject_id'].isin(selected_ids)].copy()
        print(f'  Subsampled to {len(selected_ids)} patients')

    # Normalize features fresh from THIS subset
    feature_stats = {}
    for feature in ALL_FEATURES:
        mean = df[feature].mean()
        std = df[feature].std()
        if std == 0 or pd.isna(std):
            std = 1.0
        feature_stats[feature] = {'mean': float(mean), 'std': float(std)}
        df[feature] = (df[feature] - mean) / std

    # Patient-level stratified split
    patient_ids = df['subject_id'].unique()
    patient_labels = df.groupby('subject_id')['sepsis_label'].max().loc[patient_ids]

    train_val_ids, test_ids = train_test_split(
        patient_ids, test_size=SHARED['test_size'],
        stratify=patient_labels, random_state=seed
    )
    train_val_labels = patient_labels.loc[train_val_ids]
    train_ids, val_ids = train_test_split(
        train_val_ids,
        test_size=SHARED['val_size'] / (1 - SHARED['test_size']),
        stratify=train_val_labels, random_state=seed
    )

    train_df = df[df['subject_id'].isin(train_ids)]
    val_df = df[df['subject_id'].isin(val_ids)]
    test_df = df[df['subject_id'].isin(test_ids)]

    print(f'  Split: Train={len(train_ids)} pts ({len(train_df):,} obs), '
          f'Val={len(val_ids)} pts ({len(val_df):,} obs), '
          f'Test={len(test_ids)} pts ({len(test_df):,} obs)')

    return train_df, val_df, test_df, feature_stats

## 7. Run One Experiment (Full Pipeline)

In [ ]:
def run_experiment(version, config, df_raw):
    """
    Run a single experiment: prepare data -> train -> evaluate -> save.
    Returns results dict.
    """
    print(f"\n{'='*70}")
    print(f"  EXPERIMENT {version}: {config['description']}")
    print(f"  LR={config['learning_rate']}, Hidden={config['hidden_dim']}, "
          f"Layers={config['num_layers']}, Dropout={config['dropout']}, "
          f"Focal_a={config['focal_alpha']}")
    print(f"{'='*70}")

    start_time = time.time()
    save_dir = f'{MODEL_PATH}/{version}'
    os.makedirs(save_dir, exist_ok=True)

    # --- 1. Prepare data ---
    print('\n[1/5] Preparing data...')
    train_df, val_df, test_df, feature_stats = prepare_data(
        df_raw, config['num_patients']
    )

    # Save feature stats
    with open(f'{save_dir}/feature_stats.json', 'w') as f:
        json.dump(feature_stats, f, indent=2)

    # --- 2. Create datasets ---
    print('\n[2/5] Creating datasets...')
    train_dataset = SepsisSequenceDataset(
        train_df, VITALS_FEATURES, LABS_FEATURES, SHARED['sequence_length']
    )
    val_dataset = SepsisSequenceDataset(
        val_df, VITALS_FEATURES, LABS_FEATURES, SHARED['sequence_length']
    )
    test_dataset = SepsisSequenceDataset(
        test_df, VITALS_FEATURES, LABS_FEATURES, SHARED['sequence_length']
    )

    train_loader = DataLoader(
        train_dataset, batch_size=SHARED['batch_size'],
        shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=SHARED['batch_size'],
        shuffle=False, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=SHARED['batch_size'],
        shuffle=False, num_workers=2, pin_memory=True
    )

    # --- 3. Initialize model ---
    print('\n[3/5] Initializing model...')
    model = MultiAgentSepsisPredictor(
        vitals_dim=len(VITALS_FEATURES),
        labs_dim=len(LABS_FEATURES),
        all_features_dim=len(ALL_FEATURES),
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        dropout=config['dropout']
    ).to(device)

    criterion = FocalLoss(alpha=config['focal_alpha'], gamma=SHARED['focal_gamma'])
    optimizer = optim.AdamW(
        model.parameters(), lr=config['learning_rate'],
        weight_decay=SHARED['weight_decay']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    n_params = count_parameters(model)
    print(f'  Parameters: {n_params:,}')
    print(f'  Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

    # --- 4. Training loop ---
    print(f'\n[4/5] Training (up to {SHARED["epochs"]} epochs, patience={SHARED["patience"]})...')
    best_val_auroc = 0
    patience_counter = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_auroc': [], 'val_auroc': [],
        'train_auprc': [], 'val_auprc': [],
    }

    for epoch in range(SHARED['epochs']):
        train_loss, train_auroc, train_auprc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_auroc, val_auprc, _, _, _ = evaluate(
            model, val_loader, criterion, device
        )
        scheduler.step(val_auroc)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_auroc'].append(train_auroc)
        history['val_auroc'].append(val_auroc)
        history['train_auprc'].append(train_auprc)
        history['val_auprc'].append(val_auprc)

        current_lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch+1:2d}/{SHARED["epochs"]} | '
              f'Train: {train_auroc:.4f} | Val: {val_auroc:.4f} | '
              f'AUPRC: {val_auprc:.4f} | LR: {current_lr:.1e}', end='')

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auroc': val_auroc,
                'config': config,
            }, f'{save_dir}/best_model.pt')
            print(' *BEST*')
        else:
            patience_counter += 1
            print(f' (patience {patience_counter}/{SHARED["patience"]})')
            if patience_counter >= SHARED['patience']:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # --- 5. Test evaluation ---
    print(f'\n[5/5] Evaluating on test set...')
    checkpoint = torch.load(f'{save_dir}/best_model.pt', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])

    test_loss, test_auroc, test_auprc, test_preds, test_labels, agent_weights = evaluate(
        model, test_loader, criterion, device
    )

    # Optimal F1 threshold
    prec, rec, thresholds_pr = precision_recall_curve(test_labels, test_preds)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = float(thresholds_pr[optimal_idx]) if optimal_idx < len(thresholds_pr) else 0.5
    best_f1 = float(f1_scores[optimal_idx])

    test_preds_binary = (test_preds >= optimal_threshold).astype(int)
    cm = confusion_matrix(test_labels, test_preds_binary)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    # Agent weights breakdown
    avg_weights = agent_weights.mean(axis=0)
    sepsis_mask = test_labels == 1
    sepsis_weights = agent_weights[sepsis_mask].mean(axis=0) if sepsis_mask.sum() > 0 else avg_weights
    nonsepsis_weights = agent_weights[~sepsis_mask].mean(axis=0) if (~sepsis_mask).sum() > 0 else avg_weights

    elapsed = time.time() - start_time

    # --- Print summary ---
    print(f'\n{"="*50}')
    print(f'{version} TEST RESULTS')
    print(f'{"="*50}')
    print(f'  AUROC:       {test_auroc:.4f}')
    print(f'  AUPRC:       {test_auprc:.4f}')
    print(f'  F1:          {best_f1:.4f} (threshold={optimal_threshold:.3f})')
    print(f'  Sensitivity: {sensitivity:.4f}')
    print(f'  Specificity: {specificity:.4f}')
    print(f'  PPV:         {ppv:.4f}')
    print(f'  NPV:         {npv:.4f}')
    print(f'  Confusion:   TP={tp:,} FP={fp:,} TN={tn:,} FN={fn:,}')
    print(f'  Agents:      Vitals={avg_weights[0]:.1%} Labs={avg_weights[1]:.1%} Trend={avg_weights[2]:.1%}')
    print(f'  Best epoch:  {checkpoint["epoch"]+1}')
    print(f'  Time:        {elapsed/60:.1f} min')

    # --- Save everything ---
    results = {
        'version': version,
        'description': config['description'],
        'test_auroc': float(test_auroc),
        'test_auprc': float(test_auprc),
        'test_loss': float(test_loss),
        'f1': float(best_f1),
        'optimal_threshold': optimal_threshold,
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'ppv': float(ppv),
        'npv': float(npv),
        'confusion_matrix': cm.tolist(),
        'best_epoch': int(checkpoint['epoch']),
        'n_params': n_params,
        'training_time_min': round(elapsed / 60, 1),
        'agent_weights_overall': [float(w) for w in avg_weights],
        'agent_weights_sepsis': [float(w) for w in sepsis_weights],
        'agent_weights_nonsepsis': [float(w) for w in nonsepsis_weights],
        'history': history,
        'config': {**config, **SHARED},
        'timestamp': datetime.now().isoformat(),
    }

    with open(f'{save_dir}/results.json', 'w') as f:
        json.dump(results, f, indent=2)

    np.savez(f'{save_dir}/test_predictions.npz',
             preds=test_preds, labels=test_labels, agent_weights=agent_weights)

    # Save training curves plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title(f'{version} Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['train_auroc'], label='Train'); axes[1].plot(history['val_auroc'], label='Val')
    axes[1].set_title(f'{version} AUROC'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].plot(history['train_auprc'], label='Train'); axes[2].plot(history['val_auprc'], label='Val')
    axes[2].set_title(f'{version} AUPRC'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'  Saved to {save_dir}/')

    # Cleanup GPU memory
    del model, optimizer, scheduler, criterion
    del train_dataset, val_dataset, test_dataset
    del train_loader, val_loader, test_loader
    torch.cuda.empty_cache()

    return results

## 8. RUN ALL 6 EXPERIMENTS

**Estimated time:** ~2-3 hours total on T4 GPU.

Each version saves independently to `models/<version>/`. If Colab disconnects, just comment out the finished versions and re-run this cell.

In [ ]:
# ============================================================
# CHOOSE WHICH EXPERIMENTS TO RUN
# Comment out any you've already completed to skip them
# ============================================================
VERSIONS_TO_RUN = [
    'v1',  # Baseline: 725 patients, LR=1e-3
    'v2',  # Full data, LR=1e-3 (expected to fail)
    'v3',  # Full data, LR=1e-4 (expected winner)
    'v4',  # Full data, focal alpha=0.35
    'v5',  # Full data, dropout=0.4
    'v6',  # Full data, smaller model (32/1)
]

# Uncomment below to resume after a crash:
# VERSIONS_TO_RUN = ['v4', 'v5', 'v6']  # skip v1-v3 if done

all_results = {}
total_start = time.time()

for version in VERSIONS_TO_RUN:
    config = EXPERIMENTS[version]
    results = run_experiment(version, config, df_raw)
    all_results[version] = results

total_time = (time.time() - total_start) / 60
print(f"\n\n{'='*70}")
print(f'ALL {len(VERSIONS_TO_RUN)} EXPERIMENTS COMPLETE in {total_time:.0f} minutes')
print(f"{'='*70}")

## 9. Load & Compare All Results

In [ ]:
# Load all results (works even if some were from earlier sessions)
all_results = {}
for version in ['v1', 'v2', 'v3', 'v4', 'v5', 'v6']:
    path = f'{MODEL_PATH}/{version}/results.json'
    if os.path.exists(path):
        with open(path) as f:
            all_results[version] = json.load(f)

print(f'Loaded results for: {list(all_results.keys())}\n')

# Find best
best_v = max(all_results, key=lambda v: all_results[v]['test_auroc'])
worst_v = min(all_results, key=lambda v: all_results[v]['test_auroc'])

# Summary table
print(f'{"Ver":<5} {"Pts":>6} {"LR":>8} {"H/L":>5} {"Drop":>5} {"alpha":>5} '
      f'{"AUROC":>7} {"AUPRC":>7} {"F1":>6} {"Sens":>6} {"Spec":>6} {"Time":>6} {"Status"}')
print('-' * 95)

for v in sorted(all_results.keys()):
    r = all_results[v]
    c = r['config']
    pts = c.get('num_patients') or 3559
    status = '<-- BEST' if v == best_v else ('<-- WORST' if v == worst_v else '')
    print(f'{v:<5} {pts:>6} {c["learning_rate"]:>8.0e} '
          f'{c["hidden_dim"]:>2}/{c["num_layers"]:<2} {c["dropout"]:>5} {c["focal_alpha"]:>5} '
          f'{r["test_auroc"]:>7.4f} {r["test_auprc"]:>7.4f} '
          f'{r["f1"]:>6.4f} {r["sensitivity"]:>6.3f} {r["specificity"]:>6.3f} '
          f'{r["training_time_min"]:>5.0f}m {status}')

## 10. Comparison Plots

In [ ]:
versions = sorted(all_results.keys())
aurocs = [all_results[v]['test_auroc'] for v in versions]
auprcs = [all_results[v]['test_auprc'] for v in versions]

# Color coding
colors = []
for v in versions:
    if v == best_v:
        colors.append('#27ae60')  # green
    elif v == worst_v:
        colors.append('#e74c3c')  # red
    else:
        colors.append('#3498db')  # blue

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- AUROC bar chart ---
bars = axes[0].bar(versions, aurocs, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel('AUROC', fontsize=12)
axes[0].set_title('AUROC by Experiment', fontweight='bold', fontsize=14)
axes[0].set_ylim(min(aurocs) - 0.04, max(aurocs) + 0.04)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Random')
for bar, val in zip(bars, aurocs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# --- AUPRC bar chart ---
bars = axes[1].bar(versions, auprcs, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('AUPRC', fontsize=12)
axes[1].set_title('AUPRC by Experiment', fontweight='bold', fontsize=14)
axes[1].set_ylim(min(auprcs) - 0.04, max(auprcs) + 0.04)
for bar, val in zip(bars, auprcs):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

# --- Validation AUROC curves ---
for v in versions:
    if 'history' in all_results[v]:
        h = all_results[v]['history']
        lw = 3 if v == best_v else 1.5
        axes[2].plot(h['val_auroc'], label=v, linewidth=lw)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Validation AUROC', fontsize=12)
axes[2].set_title('Training Progress', fontweight='bold', fontsize=14)
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/all_experiments_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {MODEL_PATH}/all_experiments_comparison.png')

## 11. Learning Rate Scaling Story (v1 -> v2 -> v3)

In [ ]:
if all(v in all_results for v in ['v1', 'v2', 'v3']):
    fig, ax = plt.subplots(figsize=(9, 5))

    story = [
        ('v1\n725 pts\nLR=1e-3', all_results['v1']['test_auroc'], '#27ae60'),
        ('v2\n3,559 pts\nLR=1e-3', all_results['v2']['test_auroc'], '#e74c3c'),
        ('v3\n3,559 pts\nLR=1e-4', all_results['v3']['test_auroc'], '#27ae60'),
    ]
    labels, values, clrs = zip(*story)

    bars = ax.bar(labels, values, color=clrs, edgecolor='white', linewidth=2, width=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=14)

    # Drop arrow v1->v2
    drop = (values[0] - values[1]) * 100
    ax.annotate('', xy=(1, values[1]+0.005), xytext=(0, values[0]+0.005),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax.text(0.5, (values[0]+values[1])/2 + 0.012,
            f'DROP {drop:.1f}%', ha='center', color='red', fontweight='bold', fontsize=12)

    # Recovery arrow v2->v3
    gain = (values[2] - values[1]) * 100
    ax.annotate('', xy=(2, values[2]+0.005), xytext=(1, values[1]+0.005),
                arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    ax.text(1.5, (values[1]+values[2])/2 + 0.012,
            f'RECOVERED +{gain:.1f}%', ha='center', color='green', fontweight='bold', fontsize=12)

    ax.set_ylabel('Test AUROC', fontsize=13)
    ax.set_title('Key Finding: Learning Rate Must Scale With Data Size',
                 fontweight='bold', fontsize=14)
    ax.set_ylim(min(values) - 0.05, max(values) + 0.06)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{MODEL_PATH}/lr_scaling_story.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {MODEL_PATH}/lr_scaling_story.png')
else:
    print('Need v1, v2, v3 results to generate this plot.')

## 12. Agent Weight Comparison

In [ ]:
if len(all_results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    agent_names = ['Vitals', 'Labs', 'Trend']
    agent_colors = ['#1B5E20', '#0D47A1', '#4A148C']

    # --- Overall weights per version ---
    x = np.arange(len(versions))
    width = 0.25
    for i, (agent, color) in enumerate(zip(agent_names, agent_colors)):
        weights = [all_results[v]['agent_weights_overall'][i] * 100 for v in versions]
        axes[0].bar(x + i * width, weights, width, label=f'{agent}', color=color, alpha=0.85)
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(versions)
    axes[0].set_ylabel('Weight (%)')
    axes[0].set_title('Agent Weights by Version', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # --- Sepsis vs Non-sepsis for best version ---
    bv = best_v
    sep_w = [w * 100 for w in all_results[bv]['agent_weights_sepsis']]
    non_w = [w * 100 for w in all_results[bv]['agent_weights_nonsepsis']]
    x2 = np.arange(3)
    axes[1].bar(x2 - 0.15, non_w, 0.3, label='Non-Sepsis', color='#3498db')
    axes[1].bar(x2 + 0.15, sep_w, 0.3, label='Sepsis', color='#e74c3c')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(agent_names)
    axes[1].set_ylabel('Weight (%)')
    axes[1].set_title(f'Agent Weights: Sepsis vs Non-Sepsis ({bv})', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{MODEL_PATH}/agent_weights_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {MODEL_PATH}/agent_weights_comparison.png')

## 13. ROC & PR Curves for Best Model

In [ ]:
# Load test predictions for the best version
bv = best_v
npz = np.load(f'{MODEL_PATH}/{bv}/test_predictions.npz')
preds = npz['preds']
labels = npz['labels']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC
fpr, tpr, _ = roc_curve(labels, preds)
auroc = roc_auc_score(labels, preds)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUROC = {auroc:.4f}')
axes[0].plot([0,1], [0,1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve ({bv})', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# PR
prec, rec, _ = precision_recall_curve(labels, preds)
auprc = average_precision_score(labels, preds)
axes[1].plot(rec, prec, 'g-', linewidth=2, label=f'AUPRC = {auprc:.4f}')
axes[1].axhline(y=labels.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline = {labels.mean():.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve ({bv})', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confusion matrix
threshold = all_results[bv]['optimal_threshold']
cm = confusion_matrix(labels, (preds >= threshold).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Sepsis', 'Sepsis'],
            yticklabels=['No Sepsis', 'Sepsis'])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title(f'Confusion Matrix ({bv}, t={threshold:.3f})', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/best_model_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Final Summary

In [ ]:
print('\n' + '='*70)
print('FINAL EXPERIMENT SUMMARY')
print('='*70)

print(f'\n{"Ver":<5} {"Pts":>6} {"LR":>8} {"H/L":>5} {"Drop":>5} {"a":>5} '
      f'{"AUROC":>7} {"AUPRC":>7} {"F1":>6} {"Sens":>6} {"Spec":>6} Status')
print('-' * 90)

for v in sorted(all_results.keys()):
    r = all_results[v]
    c = r['config']
    pts = c.get('num_patients') or 3559
    if v == best_v:
        status = 'WINNER'
    elif v == worst_v:
        status = 'Failed'
    else:
        status = 'Inferior'
    print(f'{v:<5} {pts:>6} {c["learning_rate"]:>8.0e} '
          f'{c["hidden_dim"]:>2}/{c["num_layers"]:<2} {c["dropout"]:>5} {c["focal_alpha"]:>5} '
          f'{r["test_auroc"]:>7.4f} {r["test_auprc"]:>7.4f} '
          f'{r["f1"]:>6.3f} {r["sensitivity"]:>6.3f} {r["specificity"]:>6.3f} {status}')

total_time = sum(r['training_time_min'] for r in all_results.values())
print(f'\nTotal training time: {total_time:.0f} minutes')
print(f'Best model: {best_v} (AUROC={all_results[best_v]["test_auroc"]:.4f})')

print(f'\nSaved files:')
for v in sorted(all_results.keys()):
    print(f'  {MODEL_PATH}/{v}/  ->  best_model.pt, results.json, test_predictions.npz, training_curves.png')
print(f'  {MODEL_PATH}/  ->  all_experiments_comparison.png, lr_scaling_story.png, agent_weights_comparison.png, best_model_curves.png')